# 1. Hausa→English data loading and preprocessing

This notebook prepares the data contract shared by our Hausa automatic speech recognition (ASR), cascade translation, and direct speech-to-text translation (S2TT) experiments. It is written for readers who know Python and basic machine learning but are new to speech data.

For every stage we identify its input, run reusable package code, inspect its output, verify an invariant, and state which later notebook consumes it. The committed notebook intentionally has no saved outputs; values marked **Live output** appear only when you run it.

## 2. Where Notebook 00 fits

1. **Notebook 00 (this file, inexpensive by default):** audit, align, split, preprocess, and define durable data artifacts.
2. **Notebook 01 (planned):** train/evaluate Hausa ASR, then feed its Hausa text to NLLB for a cascade.
3. **Notebook 02 (planned, expensive):** train Whisper on genuine Hausa-audio/English-text pairs for direct S2TT.
4. **Notebook 03 (planned, final-data access):** compare frozen zero-shot, cascade, and direct systems on the reserved held-out data.

A **source transcript** is text in the spoken language (Hausa). A **target translation** is the desired English text. ASR predicts the former; direct S2TT predicts the latter. They are different learning tasks.

## 3. Environment setup

The switches below make cost explicit. The safe path performs no training, no full-corpus download, and no inspection of NaijaS2ST `dev` target sentences. A **lazy audio** workflow keeps audio undecoded until a specific example is requested.

`REPO_REF` names the branch containing this notebook and its package changes. After merge, change it to `main`. In Colab, a missing checkout is cloned. A clean checkout is fetched and fast-forwarded. A dirty checkout is never overwritten.

In [ ]:
REPO_URL = "https://github.com/tsuxalo/Spoken-Language-Translation-Model.git"
REPO_REF = "feature/data-preprocessing-notebook"
RUN_NETWORK_SAMPLE = True
RUN_FULL_METADATA_AUDIT = False
BUILD_FULL_TRAINING_DATASET = False
WRITE_ARTIFACTS = False
SEED = 42
VALIDATION_FRACTION = 0.10
MAX_AUDIO_SECONDS = 30.0


In [ ]:
import importlib
import importlib.metadata
import subprocess
import sys
from pathlib import Path


def run(command, *, cwd=None):
    return subprocess.run(command, cwd=cwd, check=True, text=True, capture_output=True)

cwd = Path.cwd().resolve()
existing = next((p for p in (cwd, cwd.parent) if (p / ".git").exists()), None)
if existing is None:
    checkout = Path("/content/Spoken-Language-Translation-Model")
    checkout.parent.mkdir(parents=True, exist_ok=True)
    run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(checkout)])
else:
    checkout = existing
    dirty = run(["git", "status", "--porcelain"], cwd=checkout).stdout.strip()
    if dirty:
        raise RuntimeError("Existing checkout is dirty; commit/stash it or use a fresh runtime. No files were overwritten.")
    remote_branch = run(["git", "ls-remote", "--heads", "origin", REPO_REF], cwd=checkout).stdout.strip()
    local_branch = run(["git", "branch", "--list", REPO_REF], cwd=checkout).stdout.strip()
    if remote_branch:
        run(["git", "fetch", "origin", REPO_REF], cwd=checkout)
        if local_branch:
            run(["git", "switch", REPO_REF], cwd=checkout)
        else:
            run(["git", "switch", "--track", "-c", REPO_REF, f"origin/{REPO_REF}"], cwd=checkout)
        run(["git", "merge", "--ff-only", f"origin/{REPO_REF}"], cwd=checkout)
    elif local_branch:
        run(["git", "switch", REPO_REF], cwd=checkout)
        print(f"Using clean unpublished local branch {REPO_REF}; fresh Colab clone requires publication.")
    else:
        run(["git", "fetch", "origin", REPO_REF], cwd=checkout)
        run(["git", "switch", "--detach", "FETCH_HEAD"], cwd=checkout)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(checkout)], check=True)
for name in list(sys.modules):
    if name == "hausa_s2tt" or name.startswith("hausa_s2tt."):
        del sys.modules[name]
importlib.invalidate_caches()
src_path = str((checkout / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)
import hausa_s2tt

package_path = Path(hausa_s2tt.__file__).resolve()
if not package_path.is_relative_to((checkout / "src").resolve()):
    raise RuntimeError(f"Stale package import: {package_path}")
REPO_ROOT = checkout.resolve()
actual_branch = run(["git", "branch", "--show-current"], cwd=REPO_ROOT).stdout.strip() or "detached"
actual_sha = run(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT).stdout.strip()
versions = {}
for distribution in ("hausa-s2tt", "datasets", "transformers", "torch", "numpy", "soundfile"):
    try:
        versions[distribution] = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        versions[distribution] = "not installed"
print({"requested_ref": REPO_REF, "actual_branch": actual_branch, "commit": actual_sha})
print({"python": sys.version.split()[0], "package_path": str(package_path), "versions": versions})


**Verification.** The setup cell fails rather than replacing a dirty checkout, imports `hausa_s2tt` from this checkout's `src` directory, and prints the requested ref, actual branch/SHA, Python version, package path, and relevant library versions. It never prints environment variables or credentials.

## 4. Reproducibility and pinned revisions

A **dataset revision** is an immutable Git commit identifying the exact dataset snapshot. **Provenance** records where an example came from and how it was produced. The notebook reads one authoritative set of constants instead of copying hashes into cells. If a pinned resource is unavailable, downstream loaders fail; they do not silently fall back to an unversioned resource.

In [ ]:
from hausa_s2tt.revisions import (
    FLEURS_DATASET_ID,
    FLEURS_REVISION,
    HAUSA_ASR_ID,
    HAUSA_ASR_REVISION,
    NAIJA_DATASET_ID,
    NAIJA_REVISION,
    NLLB_MODEL_ID,
    NLLB_REVISION,
    WHISPER_SMALL_ID,
    WHISPER_SMALL_REVISION,
)

resources = {
    "FLEURS Hausa": (FLEURS_DATASET_ID, FLEURS_REVISION),
    "NaijaS2ST": (NAIJA_DATASET_ID, NAIJA_REVISION),
    "Whisper small": (WHISPER_SMALL_ID, WHISPER_SMALL_REVISION),
    "Hausa ASR": (HAUSA_ASR_ID, HAUSA_ASR_REVISION),
    "NLLB": (NLLB_MODEL_ID, NLLB_REVISION),
}
assert all(identifier and len(revision) == 40 for identifier, revision in resources.values())
resources


## 5. FLEURS versus NaijaS2ST

We use two datasets because they solve different parts of the problem.

| Dataset | Relationship | Project role | Split policy |
|---|---|---|---|
| FLEURS `ha_ng` | Hausa audio → Hausa transcription | Hausa ASR training, validation, and final ASR test | Preserve official train/validation/test; test is never used for model selection |
| NaijaS2ST `default` | Hausa audio aligned to English text | Supervised direct S2TT and common English references | Derive project train/validation from official train by speaker; reserve official dev for final evaluation |

FLEURS alone **is not** supervised Hausa→English S2TT data because it has no aligned English target. FLEURS documentation describes speaker separation, but its Hausa schema exposes no speaker-ID field to our code; we therefore do not claim an independently measured FLEURS speaker-overlap result. A **train/validation/test split** separates parameter learning, development/model selection, and final one-time evaluation.

## 6. Tracked metadata audits

A **metadata-only audit** checks IDs, text fields, speakers, declared durations, and alignment without decoding every waveform. This saves a large corpus download, but it does not prove that every audio file decodes. The values below are labeled **Measured output from tracked revision-matched audit**: they were measured previously and checked into `reports/`, not recomputed by this notebook run. Official NaijaS2ST `dev` appears only as aggregate held-out metadata.

In [ ]:
from hausa_s2tt.datasets import load_revision_matched_audit

naija_audit = load_revision_matched_audit(
    REPO_ROOT / "reports/naija_s2st_audit_summary.json",
    expected_dataset_id=NAIJA_DATASET_ID,
    expected_dataset_revision=NAIJA_REVISION,
)
fleurs_audit = load_revision_matched_audit(
    REPO_ROOT / "reports/fleurs_ha_ng_audit_summary.json",
    expected_dataset_id=FLEURS_DATASET_ID,
    expected_dataset_revision=FLEURS_REVISION,
)
tracked_summary = {
    "label": "Measured output from tracked revision-matched audit",
    "NaijaS2ST train": naija_audit["train"],
    "NaijaS2ST dev": {"label": "Held-out final-evaluation metadata only", **naija_audit["dev"]},
    "seed-42 project split": naija_audit["derived_seed_42_split"],
    "FLEURS official splits": fleurs_audit["splits"],
}
tracked_summary


In [ ]:
import matplotlib.pyplot as plt

train_audit = naija_audit["train"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(["Accepted", "Rejected"], [train_audit["accepted_pairs"], train_audit["rejected_hausa_rows"]])
axes[0].set(title="NaijaS2ST Hausa train pairing", ylabel="Recordings", xlabel="Audit outcome")
reasons = train_audit["rejection_reasons"]
axes[1].bar(list(reasons), list(reasons.values()))
axes[1].tick_params(axis="x", rotation=25)
axes[1].set(title="Recorded rejection reasons", ylabel="Rejected recordings", xlabel="Reason")
derived = naija_audit["derived_seed_42_split"]
names = ["Training", "Validation"]
axes[2].bar(names, [derived["training"]["pairs"], derived["validation"]["pairs"]], label="Pairs")
axes[2].set(title="Speaker-disjoint project split", ylabel="Accepted pairs", xlabel="Project split")
fig.tight_layout()
print("Interpretation: most accepted train recordings remain in project training; validation contains whole held-out speakers.")


The tracked audit reports only duration summary statistics, not a per-record duration distribution, so we do not fabricate a histogram. A full metadata opt-in later can produce that distribution. Metadata tells us whether durations, speakers, IDs, and alignment records appear valid; it does **not** prove every underlying audio file successfully decodes.

## 7. Dataset schemas

A schema lists the fields and types available to preprocessing. FLEURS provides an audio field and Hausa transcription fields. NaijaS2ST provides audio, `user_id`, `language`, language-prefixed `text_id`, `text`, duration, recording metadata, and its official split. Audio remains lazily decoded. **Live output** from the optional sample reports actual keys without displaying signed audio URLs.

In [ ]:
expected_contract_fields = {
    "FLEURS ASR input": ["audio", "raw_transcription"],
    "NaijaS2ST alignment input": ["audio", "user_id", "language", "text_id", "text", "duration", "split"],
}
expected_contract_fields


## 8. Hausa-English alignment

An **alignment key** links utterances that express the same content across languages. For example, declared Hausa `HTE_0001` and English `ETE_0001` both map to `TE_0001`. The prefix is removed only when it agrees with the row's declared language, preserving all original IDs. The input is mixed-language row metadata; the output is one accepted record per valid Hausa recording with an unambiguous English target.

In [ ]:
from hausa_s2tt.datasets import align_naija_rows, alignment_key

assert alignment_key("HTE_0001", "hausa") == "TE_0001"
assert alignment_key("ETE_0001", "english") == "TE_0001"
assert alignment_key("ETE_0001", "hausa") == "ETE_0001"  # invalid language/prefix combination is preserved
tiny_rows = [
    {"audio": {"path": "audio/e.wav"}, "language": "english", "text_id": "ETE_0001", "user_id": "E1", "text": "Good morning.", "duration": 2.0},
    {"audio": {"path": "audio/h.wav"}, "language": "hausa", "text_id": "HTE_0001", "user_id": "H1", "text": "Ina kwana.", "duration": 2.0},
]
tiny_pairs, tiny_audit = align_naija_rows(tiny_rows, split="train", dataset_revision=NAIJA_REVISION)
assert len(tiny_pairs) == 1 and tiny_pairs[0]["target_text"] == "Good morning."
{key: tiny_pairs[0][key] for key in ("source_text_id", "target_text_ids", "alignment_key", "dataset_revision")}


**Verification.** Both IDs survive alongside the canonical key, and the pinned revision is already present before any notebook-specific work. Notebook 02 will use `target_text`; Notebook 01 will use `source_text`.

## 9. Rejection rules

We reject and count missing audio, missing speaker, missing alignment key, invalid/nonpositive/over-limit duration, missing English target, conflicting targets, duplicate source records, and invalid target records. No invalid row disappears silently. A metadata-valid reference can still fail later audio decoding; those are separate checks.

In [ ]:
from hausa_s2tt.datasets import PAIRING_REJECTION_REASONS

rejection_demo = tiny_rows + [
    {"audio": {"path": "audio/missing.wav"}, "language": "hausa", "text_id": "HNO_TARGET", "user_id": "H2", "text": "Babu fassara", "duration": 2.0},
    {"audio": {"path": "audio/long.wav"}, "language": "hausa", "text_id": "HLONG", "user_id": "H3", "text": "Dogo", "duration": 31.0},
]
_, rejection_audit = align_naija_rows(rejection_demo, split="train", dataset_revision=NAIJA_REVISION)
assert set(rejection_audit.rejection_reasons) == set(PAIRING_REJECTION_REASONS)
rejection_audit.rejection_reasons


## 10. Speaker-disjoint split

A **speaker leakage** occurs when the same person's recordings appear in both training and validation. A row-level random split is unsafe because a model can benefit from the same voice's acoustic characteristics. We group accepted official-train pairs by speaker, deterministically shuffle speaker IDs with seed 42, and assign whole speakers until validation is approximately 10% of examples.

In [ ]:
from hausa_s2tt.datasets import assert_no_speaker_leakage, split_by_speaker

speaker_demo = [
    {"speaker_id": f"speaker-{speaker}", "text_id": f"row-{speaker}-{clip}", "dataset_revision": NAIJA_REVISION}
    for speaker in range(6) for clip in range(3)
]
first_split = split_by_speaker(speaker_demo, test_fraction=VALIDATION_FRACTION, seed=SEED, train_name="train", test_name="validation")
second_split = split_by_speaker(speaker_demo, test_fraction=VALIDATION_FRACTION, seed=SEED, train_name="train", test_name="validation")
assert first_split == second_split
assert_no_speaker_leakage(first_split)
{name: {"examples": len(rows), "speakers": len({row["speaker_id"] for row in rows})} for name, rows in first_split.items()}


## 11. Leakage verification and final-data guard

The checked helper requires `train speakers ∩ validation speakers = ∅`. The tracked complete audit also reports zero speaker overlap between NaijaS2ST official train and official dev. Notebook 00 does not display dev sentences, create dev training examples, run a model on dev, compute dev scores, or tune any choice on dev. Notebook 03 alone should access official dev through the existing final-evaluation guard. FLEURS test receives the same no-selection treatment for ASR.

In [ ]:
assert naija_audit["derived_seed_42_split"]["speaker_overlap"] == 0
assert naija_audit["train_dev_speaker_overlap"] == 0
final_data_policy = {
    "NaijaS2ST dev": "aggregate tracked metadata only in Notebook 00; targets reserved for Notebook 03",
    "FLEURS test": "never loaded by the training entrypoint; final ASR evaluation only",
}
final_data_policy


## 12. Audio loading (tiny public train sample)

A **sampling rate** is the number of waveform samples recorded per second. A **waveform** is amplitude over time. The default network path filters one short public Hausa row from official `train`, verifies that the dataset's current Hub commit still equals our pin before using the revision-less Viewer API, and obtains its English alignment row. Any temporary Viewer URL remains in memory only.

This cell does not touch official dev. **Live output:** safe schema keys, original sample rate, duration, and channel shape; never the audio URL.

In [ ]:
live_pair = None
raw_samples = None
raw_sample_rate = None
hausa_row = None
if RUN_NETWORK_SAMPLE:
    from urllib.error import URLError

    from huggingface_hub import HfApi

    from hausa_s2tt.audio import decode_audio
    from hausa_s2tt.datasets import (
        iter_dataset_viewer_filtered_rows,
        validate_pairing_artifact,
    )
    current_sha = HfApi().dataset_info(NAIJA_DATASET_ID).sha
    if current_sha != NAIJA_REVISION:
        raise RuntimeError(f"Viewer sample refused: current dataset SHA {current_sha} != pinned {NAIJA_REVISION}")
    try:
        hausa_rows = list(iter_dataset_viewer_filtered_rows(
            "train", where="\"language\"='hausa' AND \"duration\"<=30", limit=1
        ))
        if not hausa_rows:
            raise RuntimeError("No short Hausa train row returned by Dataset Viewer")
        hausa_row = hausa_rows[0]
        key = alignment_key(hausa_row["text_id"], hausa_row["language"])
        english_id = ("E" + key).replace("'", "''")
        english_rows = list(iter_dataset_viewer_filtered_rows(
            "train", where=f"\"language\"='english' AND \"text_id\"='{english_id}'", limit=20
        ))
        live_pairs, live_audit = align_naija_rows(
            [hausa_row, *english_rows], split="train", dataset_revision=NAIJA_REVISION
        )
        if len(live_pairs) != 1:
            raise RuntimeError(f"Expected one unambiguous live pair, got {len(live_pairs)}")
        live_pair = live_pairs[0]
        validate_pairing_artifact(live_pair)
        raw_samples, raw_sample_rate = decode_audio(hausa_row["audio"])
        print({
            "safe_row_keys": sorted(key for key in hausa_row if key != "audio"),
            "audio_representation_keys": sorted(hausa_row["audio"].keys()),
            "original_sample_rate_hz": raw_sample_rate,
            "declared_duration_seconds": hausa_row["duration"],
            "original_channel_shape": raw_samples.shape,
            "durable_audio_locator": live_pair["audio_locator"],
        })
    except (TimeoutError, URLError) as error:
        print(f"Live sample unavailable after bounded retries: {type(error).__name__}. Retry later or set RUN_NETWORK_SAMPLE=False.")
else:
    print("Live sample skipped: set RUN_NETWORK_SAMPLE=True to decode one public train clip.")


## 13. Mono conversion and 16 kHz resampling

**Mono audio** has one channel. We convert multi-channel audio by averaging channels, then resample to Whisper's required 16 kHz. The package validates nonempty finite samples and enforces the 30-second limit. The output is a one-dimensional `float32` array.

In [ ]:
mono_16k = None
if raw_samples is not None:
    import numpy as np

    from hausa_s2tt.audio import resample_audio, to_mono, validate_audio
    mono_original_rate = to_mono(raw_samples)
    mono_16k = resample_audio(mono_original_rate, raw_sample_rate, 16_000)
    mono_16k = validate_audio(mono_16k, 16_000, max_duration_seconds=MAX_AUDIO_SECONDS)
    assert mono_16k.ndim == 1 and mono_16k.dtype == np.float32
    print({"mono_shape": mono_16k.shape, "sample_rate_hz": 16_000, "dtype": str(mono_16k.dtype), "duration_seconds": mono_16k.size / 16_000})


## 14. Waveform

The waveform plot is a decode sanity check: it can reveal silence, clipping, or a surprising duration. It does not prove transcription or translation quality.

In [ ]:
if mono_16k is not None:
    import numpy as np
    time_seconds = np.arange(mono_16k.size) / 16_000
    plt.figure(figsize=(12, 3))
    plt.plot(time_seconds, mono_16k, linewidth=0.7)
    plt.title("Decoded Hausa train clip waveform")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Amplitude (float32)")
    plt.tight_layout()
    print("Interpretation: inspect the live trace for non-silent speech and obvious clipping; this is one-clip validation only.")


## 15. Whisper log-Mel features

A **log-Mel representation** describes energy over time and perceptually spaced frequency bands. Whisper's **feature extractor** converts the waveform into the model's input tensor. We load the processor at its pinned revision and read the resulting dimensions rather than hard-coding Mel bins, frames, or padding behavior. Feature extraction generally returns FP32; later training/inference may cast model inputs to BF16, FP16, or FP32 according to hardware.

In [ ]:
processor = None
input_features = None
if mono_16k is not None:
    from transformers import WhisperProcessor
    processor = WhisperProcessor.from_pretrained(
        WHISPER_SMALL_ID, revision=WHISPER_SMALL_REVISION, language="Hausa", task="translate"
    )
    feature_batch = processor.feature_extractor(mono_16k, sampling_rate=16_000, return_tensors="pt")
    input_features = feature_batch["input_features"]
    print({"tensor_shape": tuple(input_features.shape), "tensor_dtype": str(input_features.dtype), "sampling_rate_hz": processor.feature_extractor.sampling_rate})
    plt.figure(figsize=(12, 4))
    plt.imshow(input_features[0].numpy(), aspect="auto", origin="lower")
    plt.title("Whisper log-Mel input features")
    plt.xlabel("Time frame")
    plt.ylabel("Mel feature index")
    plt.colorbar(label="Log-Mel feature value")
    plt.tight_layout()
    print("Interpretation: speech energy changes across time and frequency; the tensor shape comes from the pinned processor.")


We do not precompute and commit all log-Mel arrays: that would multiply storage, duplicate reproducible derived data, reduce preprocessing flexibility, and complicate provenance. Feature extraction is inexpensive relative to storing the full derived corpus.

## 16. Source/target text preservation

Training labels retain original Unicode, capitalization, and punctuation. Hausa characters such as `ɗ`, `ƙ`, and `ɓ` must survive round trips. Whitespace cleanup is stored as a separate field; it does not overwrite the original. This is **training-label preservation**. Evaluation normalization for WER/CER is a separate metric-time operation and must use language-appropriate rules—never Hausa rules on English or English rules on Hausa.

In [ ]:
unicode_probe = "  Ɗan ƙauye ya ce ɓera.  "
whitespace_copy = " ".join(unicode_probe.split())
assert all(character in unicode_probe.casefold() for character in "ɗƙɓ")
assert unicode_probe != whitespace_copy
text_demo = {
    "constructed_unicode_probe_original": unicode_probe,
    "constructed_unicode_probe_whitespace_copy": whitespace_copy,
    "live_source_text": None if live_pair is None else live_pair["source_text"],
    "live_english_target": None if live_pair is None else live_pair["target_text"],
}
text_demo


## 17. English target tokenization and loss masking

A **tokenizer** converts an English target string into decoder token IDs. A padded batch gives sequences a common length. The existing training collator then replaces padding IDs with `-100`, the ignore index used by PyTorch cross-entropy loss, so padding does not teach the model to predict meaningless tokens.

English string → Whisper tokenizer → token IDs → padded batch → padding replaced by `-100`.

In [ ]:
if processor is not None and live_pair is not None:
    from hausa_s2tt.training import SpeechSeq2SeqCollator
    english_target = live_pair["target_text"]
    tokens = processor.tokenizer(english_target)
    collator = SpeechSeq2SeqCollator(processor=processor, target_column="target_text")
    batch = collator([
        {"audio": {"array": mono_16k, "sampling_rate": 16_000}, "target_text": english_target},
        {"audio": {"array": mono_16k, "sampling_rate": 16_000}, "target_text": "Hi."},
        {"audio": {"array": mono_16k, "sampling_rate": 16_000}, "target_text": "This deliberately long constructed English target demonstrates padding and loss masking across a batch."},
    ])
    masked_padding_count = int((batch["labels"] == -100).sum())
    assert masked_padding_count > 0
    print({"target": english_target, "token_ids": tokens["input_ids"], "labels_shape": tuple(batch["labels"].shape), "masked_padding_count": masked_padding_count})


## 18. Training-example schema

The durable JSONL contract below contains coordinates for recovering audio, never waveform arrays, bytes, bearer tokens, or signed URLs. `audio_locator.split` remains the official source split, while top-level `split` becomes project `train` or `validation` after speaker splitting. Notebook 01 consumes Hausa audio plus `source_text`; Notebook 02 consumes the same audio plus English `target_text`.

In [ ]:
from hausa_s2tt.datasets import (
    ARTIFACT_SCHEMA_VERSION,
    PAIRING_ARTIFACT_REQUIRED_FIELDS,
    pairing_artifact_record,
)

schema_contract = {field: "required" for field in PAIRING_ARTIFACT_REQUIRED_FIELDS}
safe_example = pairing_artifact_record(tiny_pairs[0])
assert "audio" not in safe_example
assert safe_example["dataset_revision"] == NAIJA_REVISION
assert safe_example["artifact_schema_version"] == ARTIFACT_SCHEMA_VERSION
{"schema": schema_contract, "safe_example": safe_example}


## 19. Artifact generation (expensive opt-in)

The complete tracked metadata audit took about 199 seconds on its recorded machine and excluded audio bytes. The dataset's recorded total footprint is roughly 69.9 GB, so `BUILD_FULL_TRAINING_DATASET=True` can require substantial download/cache storage and should be preceded by a local disk/compute check. Neither expensive path runs by default.

When explicitly enabled, metadata audits go under `artifacts/audits/naija_s2st/`; training JSONL and its manifest go under `artifacts/data/naija_s2st/`. Both directories are ignored by Git. The manifest records schema version, timestamp, Git SHA, dataset/revision, split policy, seed, duration limit, and counts.

In [ ]:
full_pairs = None
full_audit = None
if RUN_FULL_METADATA_AUDIT:
    from hausa_s2tt.datasets import iter_dataset_parquet_metadata
    train_metadata = list(iter_dataset_parquet_metadata("train", workers=4))
    full_pairs, full_audit = align_naija_rows(
        train_metadata, split="train", max_duration_seconds=MAX_AUDIO_SECONDS, dataset_revision=NAIJA_REVISION
    )
    print({"accepted": full_audit.paired_examples, "rejected": full_audit.rejected_records, "audio_decode_validation": "not performed"})
else:
    print("Full metadata audit skipped (default). Tracked revision-matched results were used above.")


In [ ]:
project_datasets = None
project_audits = None
if BUILD_FULL_TRAINING_DATASET:
    from hausa_s2tt.datasets import (
        PairingAudit,
        load_naija_split,
        pair_naija_dataset,
        split_dataset_by_speaker,
    )
    raw_train = load_naija_split("train", revision=NAIJA_REVISION, sampling_rate=16_000)
    paired_train, official_train_audit = pair_naija_dataset(
        raw_train, split="train", max_duration_seconds=MAX_AUDIO_SECONDS, dataset_revision=NAIJA_REVISION
    )
    project_datasets = split_dataset_by_speaker(
        paired_train, test_fraction=VALIDATION_FRACTION, seed=SEED, train_name="train", test_name="validation"
    )
    assert not set(project_datasets["train"]["speaker_id"]) & set(project_datasets["validation"]["speaker_id"])
    project_audits = [PairingAudit(split=name, paired_examples=len(dataset)) for name, dataset in project_datasets.items()]
    print({name: len(dataset) for name, dataset in project_datasets.items()})
else:
    print("Full training-dataset build skipped (default).")


In [ ]:
if WRITE_ARTIFACTS:
    if project_datasets is None or project_audits is None:
        raise RuntimeError("Set BUILD_FULL_TRAINING_DATASET=True before WRITE_ARTIFACTS=True")
    from hausa_s2tt.datasets import (
        build_pairing_manifest,
        current_git_commit,
        write_pairing_artifacts,
        write_pairing_manifest,
    )
    output_dir = REPO_ROOT / "artifacts/data/naija_s2st"
    for audit in project_audits:
        write_pairing_artifacts(project_datasets[audit.split], audit, output_dir)
    manifest = build_pairing_manifest(
        project_audits, git_commit=current_git_commit(REPO_ROOT), dataset_revision=NAIJA_REVISION,
        split_policy="NaijaS2ST official train partitioned by Hausa speaker; official dev reserved",
        seed=SEED, max_duration_seconds=MAX_AUDIO_SECONDS,
    )
    manifest_path = write_pairing_manifest(manifest, output_dir)
    print({"artifact_directory": str(output_dir), "manifest": str(manifest_path)})
else:
    print("Artifact writing skipped (default). No generated JSONL was added to the repository.")


## 20. Interfaces to later notebooks

| Consumer | Inputs from Notebook 00 | Outputs it should create |
|---|---|---|
| Notebook 01 — ASR/cascade | `audio_locator`, Hausa `source_text`, project split, speaker/provenance | Hausa ASR predictions/metrics; English cascade predictions/metrics; latency |
| Notebook 02 — direct S2TT | `audio_locator`, genuine English `target_text`, project train/validation, speaker/provenance, manifest | Checkpoints, configs, telemetry, validation predictions |
| Notebook 03 — final comparison | Frozen predictions/checkpoints/configs plus official held-out data loaded through the final guard | Per-example predictions, aggregate ASR/translation metrics, efficiency comparison, limitations |

Later notebooks should resolve audio from dataset ID + immutable revision + official split + row index. They should verify `artifact_schema_version` before consuming a file.

## 21. Ethics, licensing, consent, and limitations

[FLEURS](https://huggingface.co/datasets/google/fleurs) and [NaijaS2ST](https://huggingface.co/datasets/McGill-NLP/NaijaS2ST) are documented as CC BY 4.0; downstream work must retain attribution and review the current dataset cards plus this repository's `reports/SOURCE_VERIFICATION.md`. A permissive data license does not establish that we independently verified every speaker's consent context. Coverage can be uneven across Hausa dialects, regions, genders, speakers, topics, and recording conditions. Speaker imbalance and microphone/noise differences may produce misleading aggregate performance. Low-resource-language systems can amplify data errors; translation systems can omit, distort, or hallucinate content. Durable provenance supports audits but does not remove these risks.

This is a research prototype. It must not be the sole translation source for medical decisions, legal proceedings, immigration, policing, government benefits, or any other consequential decision.

## 22. Summary and next steps

We distinguished ASR data from translation data, verified tracked reports against pinned revisions, demonstrated alignment and explicit rejection accounting, produced a deterministic speaker-disjoint split, decoded at most one short official-train clip, generated actual Whisper features and English labels when the live sample is enabled, and defined a safe versioned artifact contract.

The next notebook should consume project-train/project-validation Hausa audio plus `source_text`, train or load the Hausa ASR diagnostic, evaluate WER/CER on validation, then translate the same ASR hypotheses with pinned NLLB `hau_Latn → eng_Latn`. It must keep ASR and translation metrics separate and leave NaijaS2ST official dev untouched for Notebook 03.